# Step 4 — Router
**Phase 3 | NIKKO Engineering Collective**

Tests `agents/router.py`. Pure deterministic logic — no model required.

**What we are testing:**
1. Active/acute risk → CRISIS unconditionally (REQ-200-124/125)
2. distress_level=crisis (no explicit risk key) → CRISIS
3. Signal confidence < 0.40 → COMFORT fallback (REQ-000-F01)
4. Guidance intent signals → GUIDANCE
5. Default path → COMFORT
6. Passive risk present but not crisis → flagged, mode not elevated
7. crisis_override invariant enforced by Pydantic
8. Loop limit: attempt_count > 2 raises ValueError (REQ-200-170)


In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from agents.router import Router, RouterDecision
from agents.signal_agent import SignalAgent
from docs.schemas.acp_schemas import DistressLevel, OperationalMode

router = Router()
sa = SignalAgent()  # used only for mock_analyze — no model load
print('Router and SignalAgent instantiated.')

### Helper


In [ ]:
def make_signal(distress='moderate', risk=None, support=None, behavioral=None, confidence=0.75):
    """Build a SignalPayload via mock_analyze for test cases."""
    return sa.mock_analyze({
        'distress_level': distress,
        'emotional_states': ['sadness_spectrum.low_mood_language'],
        'cognitive_patterns': [],
        'behavioral_indicators': behavioral or [],
        'risk_indicators': risk or [],
        'support_needs': support or ['emotional_validation'],
        'confidence': confidence,
        'uncertainty_notes': ''
    })

print('Helper ready.')

## Test cases


In [ ]:
# --- T1: Active risk → CRISIS (REQ-200-124) ---
signal = make_signal(
    distress='high',
    risk=['risk.active.suicidal_ideation', 'risk.active.preparation_statement'],
    support=['crisis_escalation']
)
d = router.route(signal)
ok = d.mode == OperationalMode.CRISIS and d.crisis_override is True
print(f'T1 active risk -> CRISIS         : {chr(10003) if ok else chr(10007)+" FAIL"}')
print(f'   mode={d.mode.value}  crisis_override={d.crisis_override}')
print(f'   rationale: {d.routing_rationale[:80]}')

In [ ]:
# --- T2: Acute risk → CRISIS (REQ-200-125) ---
signal = make_signal(
    distress='crisis',
    risk=['risk.acute.intent_language', 'risk.acute.immediacy_statement'],
    support=['crisis_escalation']
)
d = router.route(signal)
ok = d.mode == OperationalMode.CRISIS and d.crisis_override is True
print(f'T2 acute risk -> CRISIS          : {chr(10003) if ok else chr(10007)+" FAIL"}')
print(f'   mode={d.mode.value}  crisis_override={d.crisis_override}')

In [ ]:
# --- T3: distress=crisis, no explicit risk key → CRISIS ---
signal = make_signal(distress='crisis', risk=[], support=['crisis_escalation'])
d = router.route(signal)
ok = d.mode == OperationalMode.CRISIS
print(f'T3 distress=crisis, no risk key  : {chr(10003) if ok else chr(10007)+" FAIL"}')
print(f'   mode={d.mode.value}  rationale: {d.routing_rationale[:80]}')

In [ ]:
# --- T4: Low confidence → COMFORT fallback (REQ-000-F01) ---
signal = make_signal(distress='moderate', confidence=0.25)
d = router.route(signal)
ok = d.mode == OperationalMode.COMFORT and d.crisis_override is False
print(f'T4 conf=0.25 -> COMFORT fallback : {chr(10003) if ok else chr(10007)+" FAIL"}')
print(f'   mode={d.mode.value}  confidence={d.confidence}')

In [ ]:
# --- T5: Guidance intent (psychoeducation need) → GUIDANCE ---
signal = make_signal(
    distress='moderate',
    support=['emotional_validation', 'psychoeducation'],
    confidence=0.78
)
d = router.route(signal)
ok = d.mode == OperationalMode.GUIDANCE and d.crisis_override is False
print(f'T5 psychoeducation need -> GUID  : {chr(10003) if ok else chr(10007)+" FAIL"}')
print(f'   mode={d.mode.value}  rationale: {d.routing_rationale[:80]}')

In [ ]:
# --- T6: Guidance intent (help_seeking_behavior) → GUIDANCE ---
signal = make_signal(
    distress='low',
    behavioral=['help_seeking_behavior'],
    support=['emotional_validation'],
    confidence=0.65
)
d = router.route(signal)
ok = d.mode == OperationalMode.GUIDANCE
print(f'T6 help_seeking_behavior -> GUID : {chr(10003) if ok else chr(10007)+" FAIL"}')
print(f'   mode={d.mode.value}')

In [ ]:
# --- T7: No crisis, no guidance intent → COMFORT (default) ---
signal = make_signal(
    distress='moderate',
    support=['emotional_validation', 'grounding_stabilization'],
    confidence=0.70
)
d = router.route(signal)
ok = d.mode == OperationalMode.COMFORT
print(f'T7 validation need -> COMFORT    : {chr(10003) if ok else chr(10007)+" FAIL"}')
print(f'   mode={d.mode.value}')

In [ ]:
# --- T8: Passive risk present — mode NOT elevated, but flag set ---
# Passive risk alone does not trigger CRISIS. (REQ-100-PR1)
signal = make_signal(
    distress='high',
    risk=['risk.passive.wishing_to_disappear'],
    support=['emotional_validation'],
    confidence=0.72
)
d = router.route(signal)
ok_mode = d.mode == OperationalMode.COMFORT   # not crisis
ok_flag = d.passive_risk_flag is True          # but flagged
ok = ok_mode and ok_flag
print(f'T8 passive risk -> not CRISIS, flagged: {chr(10003) if ok else chr(10007)+" FAIL"}')
print(f'   mode={d.mode.value}  passive_risk_flag={d.passive_risk_flag}')

In [ ]:
# --- T9: Loop limit — attempt_count > 2 raises ValueError (REQ-200-170) ---
from pydantic import ValidationError

signal = make_signal(distress='low')
try:
    router.route(signal, attempt_count=3)
    print(f'T9 loop limit: {chr(10007)} FAIL — should have raised ValueError')
except ValueError as e:
    print(f'T9 loop limit -> ValueError     : {chr(10003)}')
    print(f'   {str(e)[:90]}')

In [ ]:
# --- T10: crisis_override invariant (Pydantic model_validator) ---
# Manually constructing a RouterDecision with mode=CRISIS but crisis_override=False
# must be rejected by Pydantic.
try:
    bad = RouterDecision(
        mode=OperationalMode.CRISIS,
        routing_rationale='test',
        confidence=0.90,
        crisis_override=False,   # WRONG
        attempt_count=1
    )
    print(f'T10 invariant: {chr(10007)} FAIL — should have raised ValidationError')
except Exception as e:
    print(f'T10 crisis_override invariant   : {chr(10003)}')
    print(f'   {str(e)[:90]}')

In [ ]:
# --- Summary ---
print('All Router tests complete.')
print('If all cells show checkmarks, commit and move to Step 5.')

---
## Step 4 complete
Next: Step 5 — `agents/support_strategy_agent.py`
